In [64]:
#import SparkSession
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('log_reg').getOrCreate()

In [65]:
#read the dataset
df=spark.read.csv('Log_Reg_dataset.csv',inferSchema=True,header=True)

In [66]:
from pyspark.sql.functions import *


In [67]:
#check the shape of the data 
print((df.count(),len(df.columns)))

(20000, 6)


In [68]:
#printSchema
df.printSchema()

root
 |-- Country: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Repeat_Visitor: integer (nullable = true)
 |-- Search_Engine: string (nullable = true)
 |-- Web_pages_viewed: integer (nullable = true)
 |-- Status: integer (nullable = true)



In [69]:
#number of columns in dataset
df.columns

['Country',
 'Age',
 'Repeat_Visitor',
 'Search_Engine',
 'Web_pages_viewed',
 'Status']

In [70]:
#view the dataset
df.show(5)

+---------+---+--------------+-------------+----------------+------+
|  Country|Age|Repeat_Visitor|Search_Engine|Web_pages_viewed|Status|
+---------+---+--------------+-------------+----------------+------+
|    India| 41|             1|        Yahoo|              21|     1|
|   Brazil| 28|             1|        Yahoo|               5|     0|
|   Brazil| 40|             0|       Google|               3|     0|
|Indonesia| 31|             1|         Bing|              15|     1|
| Malaysia| 32|             0|       Google|              15|     1|
+---------+---+--------------+-------------+----------------+------+
only showing top 5 rows


In [71]:
#Exploratory Data Analysis
df.describe().show()


+-------+--------+-----------------+-----------------+-------------+-----------------+------------------+
|summary| Country|              Age|   Repeat_Visitor|Search_Engine| Web_pages_viewed|            Status|
+-------+--------+-----------------+-----------------+-------------+-----------------+------------------+
|  count|   20000|            20000|            20000|        20000|            20000|             20000|
|   mean|    NULL|         28.53955|           0.5029|         NULL|           9.5533|               0.5|
| stddev|    NULL|7.888912950773227|0.500004090187782|         NULL|6.073903499824976|0.5000125004687693|
|    min|  Brazil|               17|                0|         Bing|                1|                 0|
|    max|Malaysia|              111|                1|        Yahoo|               29|                 1|
+-------+--------+-----------------+-----------------+-------------+-----------------+------------------+



In [72]:
df.groupBy('Country').count().show()

+---------+-----+
|  Country|count|
+---------+-----+
| Malaysia| 1218|
|    India| 4018|
|Indonesia|12178|
|   Brazil| 2586|
+---------+-----+



In [73]:
df.groupBy('Search_Engine').count().show()

+-------------+-----+
|Search_Engine|count|
+-------------+-----+
|        Yahoo| 9859|
|         Bing| 4360|
|       Google| 5781|
+-------------+-----+



In [74]:
df.groupBy('Status').count().show()

+------+-----+
|Status|count|
+------+-----+
|     1|10000|
|     0|10000|
+------+-----+



In [75]:
df.groupBy('Country').mean().show()

+---------+------------------+-------------------+---------------------+--------------------+
|  Country|          avg(Age)|avg(Repeat_Visitor)|avg(Web_pages_viewed)|         avg(Status)|
+---------+------------------+-------------------+---------------------+--------------------+
| Malaysia|27.792282430213465| 0.5730706075533661|   11.192118226600986|  0.6568144499178982|
|    India|27.976854156296664| 0.5433051269288203|   10.727227476356397|  0.6212045793927327|
|Indonesia| 28.43159796354081| 0.5207751683363442|    9.985711939563148|  0.5422893742814913|
|   Brazil|30.274168600154677|  0.322892498066512|    4.921113689095128|0.038669760247486466|
+---------+------------------+-------------------+---------------------+--------------------+



In [76]:
df.groupBy('Search_Engine').mean().show()

+-------------+------------------+-------------------+---------------------+------------------+
|Search_Engine|          avg(Age)|avg(Repeat_Visitor)|avg(Web_pages_viewed)|       avg(Status)|
+-------------+------------------+-------------------+---------------------+------------------+
|        Yahoo|28.569226087838523| 0.5094837204584644|    9.599655137437875|0.5071508266558474|
|         Bing| 28.68394495412844| 0.4720183486238532|    9.114908256880733|0.4559633027522936|
|       Google|28.380038055699707| 0.5149628092025601|    9.804878048780488|0.5210171250648676|
+-------------+------------------+-------------------+---------------------+------------------+



In [77]:
df.groupBy('Status').mean().show()

+------+--------+-------------------+---------------------+-----------+
|Status|avg(Age)|avg(Repeat_Visitor)|avg(Web_pages_viewed)|avg(Status)|
+------+--------+-------------------+---------------------+-----------+
|     1| 26.5435|             0.7019|              14.5617|        1.0|
|     0| 30.5356|             0.3039|               4.5449|        0.0|
+------+--------+-------------------+---------------------+-----------+



In [78]:
#converting categorical data to numerical form

In [79]:
#import required libraries

from pyspark.ml.feature import StringIndexer


In [80]:
#Indexing 
'''The first step is to label the column using StringIndexer into
numerical form. It allocates unique values to each of the categories of the
column. So, in the below example, all of the three values of search engine
(Yahoo, Google, Bing) are assigned values (0.0,1.0,2.0). This is visible in the
column named search_engine_num.'''

'The first step is to label the column using StringIndexer into\nnumerical form. It allocates unique values to each of the categories of the\ncolumn. So, in the below example, all of the three values of search engine\n(Yahoo, Google, Bing) are assigned values (0.0,1.0,2.0). This is visible in the\ncolumn named search_engine_num.'

In [81]:
search_engine_indexer = StringIndexer(inputCol="Search_Engine", outputCol="Search_Engine_Num").fit(df)
df = search_engine_indexer.transform(df)
'''Yahoo → 0.0
    Google → 1.0
    Bing → 2.0
'''

'Yahoo → 0.0\n    Google → 1.0\n    Bing → 2.0\n'

In [82]:
df.show(30,False)

+---------+---+--------------+-------------+----------------+------+-----------------+
|Country  |Age|Repeat_Visitor|Search_Engine|Web_pages_viewed|Status|Search_Engine_Num|
+---------+---+--------------+-------------+----------------+------+-----------------+
|India    |41 |1             |Yahoo        |21              |1     |0.0              |
|Brazil   |28 |1             |Yahoo        |5               |0     |0.0              |
|Brazil   |40 |0             |Google       |3               |0     |1.0              |
|Indonesia|31 |1             |Bing         |15              |1     |2.0              |
|Malaysia |32 |0             |Google       |15              |1     |1.0              |
|Brazil   |32 |0             |Google       |3               |0     |1.0              |
|Brazil   |32 |0             |Google       |6               |0     |1.0              |
|Indonesia|27 |0             |Google       |9               |0     |1.0              |
|Indonesia|32 |0             |Yahoo        

In [83]:
'''A one-hot encoder that maps a column of category indices to a column of binary vectors,
 with at most a single one-value per row that indicates the input category index. For example with 5 categories,
   an input value of 2.0 would map to an output vector of [0.0, 0.0, 1.0, 0.0]. The last category is 
   not included by default (configurable via dropLast), because it makes the vector entries sum up to one,
     and hence linearly dependent. So an input value of 4.0 maps to [0.0, 0.0, 0.0, 0.0].'''
from pyspark.ml.feature import OneHotEncoder

In [84]:
#one hot encoding
search_engine_encoder = OneHotEncoder(inputCol="Search_Engine_Num", outputCol="Search_Engine_Vector")
df = search_engine_encoder.fit(df).transform(df)

In [85]:
df.show(30,False)
'''Yahoo → 0.0
    Google → 1.0
    Bing → 2.0

#Search_Engine_Vector (size, indices_of_1s, values)
(2, [1], [1.0])  
    2 → vector length = 2
    [1] → the “1” is located at index 1
    [1.0] → the value is 1.0
    That represents the full vector: [0.0, 1.0] 
        [0,0]→ [0,0,1]

If index = 0
 Yahoo → [1.0, 0.0] → (2, [0], [1.0])

If index = 1

Google → [0.0, 1.0] → (2, [1], [1.0])

If index = 2

Bing (the last category) → [0.0, 0.0]  → (2,[],[])

'''

+---------+---+--------------+-------------+----------------+------+-----------------+--------------------+
|Country  |Age|Repeat_Visitor|Search_Engine|Web_pages_viewed|Status|Search_Engine_Num|Search_Engine_Vector|
+---------+---+--------------+-------------+----------------+------+-----------------+--------------------+
|India    |41 |1             |Yahoo        |21              |1     |0.0              |(2,[0],[1.0])       |
|Brazil   |28 |1             |Yahoo        |5               |0     |0.0              |(2,[0],[1.0])       |
|Brazil   |40 |0             |Google       |3               |0     |1.0              |(2,[1],[1.0])       |
|Indonesia|31 |1             |Bing         |15              |1     |2.0              |(2,[],[])           |
|Malaysia |32 |0             |Google       |15              |1     |1.0              |(2,[1],[1.0])       |
|Brazil   |32 |0             |Google       |3               |0     |1.0              |(2,[1],[1.0])       |
|Brazil   |32 |0            

'Yahoo → 0.0\n    Google → 1.0\n    Bing → 2.0\n\n#Search_Engine_Vector (size, indices_of_1s, values)\n(2, [1], [1.0])  \n    2 → vector length = 2\n    [1] → the “1” is located at index 1\n    [1.0] → the value is 1.0\n    That represents the full vector: [0.0, 1.0] \n        [0,0]→ [0,0,1]\n\nIf index = 0\n Yahoo → [1.0, 0.0] → (2, [0], [1.0])\n\nIf index = 1\n\nGoogle → [0.0, 1.0] → (2, [1], [1.0])\n\nIf index = 2\n\nBing (the last category) → [0.0, 0.0]  → (2,[],[])\n\n'

In [86]:
df.groupBy('Search_Engine').count().orderBy('count',ascending=False).show(5,False)

+-------------+-----+
|Search_Engine|count|
+-------------+-----+
|Yahoo        |9859 |
|Google       |5781 |
|Bing         |4360 |
+-------------+-----+



In [87]:
df.groupBy('Search_Engine_Num').count().orderBy('count',ascending=False).show(5,False)

+-----------------+-----+
|Search_Engine_Num|count|
+-----------------+-----+
|0.0              |9859 |
|1.0              |5781 |
|2.0              |4360 |
+-----------------+-----+



In [88]:
df.groupBy('Search_Engine_Vector').count().orderBy('count',ascending=False).show(5,False)

+--------------------+-----+
|Search_Engine_Vector|count|
+--------------------+-----+
|(2,[0],[1.0])       |9859 |
|(2,[1],[1.0])       |5781 |
|(2,[],[])           |4360 |
+--------------------+-----+



In [89]:
country_indexer = StringIndexer(inputCol="Country", outputCol="Country_Num").fit(df)
df = country_indexer.transform(df)

In [90]:
df.select(['Country','Country_Num']).show(10,False)

+---------+-----------+
|Country  |Country_Num|
+---------+-----------+
|India    |1.0        |
|Brazil   |2.0        |
|Brazil   |2.0        |
|Indonesia|0.0        |
|Malaysia |3.0        |
|Brazil   |2.0        |
|Brazil   |2.0        |
|Indonesia|0.0        |
|Indonesia|0.0        |
|Indonesia|0.0        |
+---------+-----------+
only showing top 10 rows


In [91]:
#one hot encoding
country_encoder = OneHotEncoder(inputCol="Country_Num", outputCol="Country_Vector")
df = country_encoder.fit(df).transform(df)

In [92]:
df.select(['Country','country_Num','Country_Vector']).show(10,False)

+---------+-----------+--------------+
|Country  |country_Num|Country_Vector|
+---------+-----------+--------------+
|India    |1.0        |(3,[1],[1.0]) |
|Brazil   |2.0        |(3,[2],[1.0]) |
|Brazil   |2.0        |(3,[2],[1.0]) |
|Indonesia|0.0        |(3,[0],[1.0]) |
|Malaysia |3.0        |(3,[],[])     |
|Brazil   |2.0        |(3,[2],[1.0]) |
|Brazil   |2.0        |(3,[2],[1.0]) |
|Indonesia|0.0        |(3,[0],[1.0]) |
|Indonesia|0.0        |(3,[0],[1.0]) |
|Indonesia|0.0        |(3,[0],[1.0]) |
+---------+-----------+--------------+
only showing top 10 rows


In [93]:
df.groupBy('Country').count().orderBy('count',ascending=False).show(5,False)

+---------+-----+
|Country  |count|
+---------+-----+
|Indonesia|12178|
|India    |4018 |
|Brazil   |2586 |
|Malaysia |1218 |
+---------+-----+



In [94]:
df.groupBy('Country_Num').count().orderBy('count',ascending=False).show(5,False)

+-----------+-----+
|Country_Num|count|
+-----------+-----+
|0.0        |12178|
|1.0        |4018 |
|2.0        |2586 |
|3.0        |1218 |
+-----------+-----+



In [95]:
df.groupBy('Country_Vector').count().orderBy('count',ascending=False).show(5,False)

+--------------+-----+
|Country_Vector|count|
+--------------+-----+
|(3,[0],[1.0]) |12178|
|(3,[1],[1.0]) |4018 |
|(3,[2],[1.0]) |2586 |
|(3,[],[])     |1218 |
+--------------+-----+



In [96]:
df.show()

+---------+---+--------------+-------------+----------------+------+-----------------+--------------------+-----------+--------------+
|  Country|Age|Repeat_Visitor|Search_Engine|Web_pages_viewed|Status|Search_Engine_Num|Search_Engine_Vector|Country_Num|Country_Vector|
+---------+---+--------------+-------------+----------------+------+-----------------+--------------------+-----------+--------------+
|    India| 41|             1|        Yahoo|              21|     1|              0.0|       (2,[0],[1.0])|        1.0| (3,[1],[1.0])|
|   Brazil| 28|             1|        Yahoo|               5|     0|              0.0|       (2,[0],[1.0])|        2.0| (3,[2],[1.0])|
|   Brazil| 40|             0|       Google|               3|     0|              1.0|       (2,[1],[1.0])|        2.0| (3,[2],[1.0])|
|Indonesia| 31|             1|         Bing|              15|     1|              2.0|           (2,[],[])|        0.0| (3,[0],[1.0])|
| Malaysia| 32|             0|       Google|           

In [97]:
from pyspark.ml.feature import VectorAssembler

In [98]:
df_assembler = VectorAssembler(inputCols=['Search_Engine_vector','country_vector','Age', 'Repeat_Visitor','Web_pages_viewed'], outputCol="features")
df = df_assembler.transform(df)

In [99]:
df.printSchema()

root
 |-- Country: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Repeat_Visitor: integer (nullable = true)
 |-- Search_Engine: string (nullable = true)
 |-- Web_pages_viewed: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Search_Engine_Num: double (nullable = false)
 |-- Search_Engine_Vector: vector (nullable = true)
 |-- Country_Num: double (nullable = false)
 |-- Country_Vector: vector (nullable = true)
 |-- features: vector (nullable = true)



In [100]:
df.select(['features','Status']).show(10,False)

+-----------------------------------+------+
|features                           |Status|
+-----------------------------------+------+
|[1.0,0.0,0.0,1.0,0.0,41.0,1.0,21.0]|1     |
|[1.0,0.0,0.0,0.0,1.0,28.0,1.0,5.0] |0     |
|(8,[1,4,5,7],[1.0,1.0,40.0,3.0])   |0     |
|(8,[2,5,6,7],[1.0,31.0,1.0,15.0])  |1     |
|(8,[1,5,7],[1.0,32.0,15.0])        |1     |
|(8,[1,4,5,7],[1.0,1.0,32.0,3.0])   |0     |
|(8,[1,4,5,7],[1.0,1.0,32.0,6.0])   |0     |
|(8,[1,2,5,7],[1.0,1.0,27.0,9.0])   |0     |
|(8,[0,2,5,7],[1.0,1.0,32.0,2.0])   |0     |
|(8,[2,5,6,7],[1.0,31.0,1.0,16.0])  |1     |
+-----------------------------------+------+
only showing top 10 rows


In [101]:
#select data for building model
model_df=df.select(['features','Status'])

In [102]:
from pyspark.ml.classification import LogisticRegression

In [103]:
#split the data 
training_df,test_df=model_df.randomSplit([0.75,0.25])

In [104]:
training_df.count()

14998

In [105]:
training_df.groupBy('Status').count().show()

+------+-----+
|Status|count|
+------+-----+
|     1| 7514|
|     0| 7484|
+------+-----+



In [106]:
test_df.count()

5002

In [107]:
test_df.groupBy('Status').count().show()

+------+-----+
|Status|count|
+------+-----+
|     1| 2486|
|     0| 2516|
+------+-----+



In [ ]:
log_reg=LogisticRegression(labelCol='Status').fit(training_df)

25/11/24 17:19:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [47]:
#Training Results

In [48]:
train_results=log_reg.evaluate(training_df).predictions

In [49]:
train_results.filter(train_results['Status']==1).filter(train_results['prediction']==1).select(['Status','prediction','probability']).show(10,False)

+------+----------+----------------------------------------+
|Status|prediction|probability                             |
+------+----------+----------------------------------------+
|1     |1.0       |[0.2995455582465126,0.7004544417534875] |
|1     |1.0       |[0.2995455582465126,0.7004544417534875] |
|1     |1.0       |[0.2995455582465126,0.7004544417534875] |
|1     |1.0       |[0.16689163833247017,0.8331083616675299]|
|1     |1.0       |[0.16689163833247017,0.8331083616675299]|
|1     |1.0       |[0.16689163833247017,0.8331083616675299]|
|1     |1.0       |[0.08578864167645286,0.9142113583235472]|
|1     |1.0       |[0.08578864167645286,0.9142113583235472]|
|1     |1.0       |[0.08578864167645286,0.9142113583235472]|
|1     |1.0       |[0.08578864167645286,0.9142113583235472]|
+------+----------+----------------------------------------+
only showing top 10 rows


Probability at 0 index is for 0 class and probabilty as 1 index is for 1 class

In [50]:
correct_preds=train_results.filter(train_results['Status']==1).filter(train_results['prediction']==1).count()


In [51]:
training_df.filter(training_df['Status']==1).count()

7515

In [52]:
#accuracy on training dataset 
float(correct_preds)/(training_df.filter(training_df['Status']==1).count())

0.9391882900864936

In [53]:
#Test Set results

In [54]:
results=log_reg.evaluate(test_df).predictions

In [55]:
results.select(['Status','prediction']).show(10,False)

+------+----------+
|Status|prediction|
+------+----------+
|0     |0.0       |
|0     |0.0       |
|0     |0.0       |
|0     |0.0       |
|0     |0.0       |
|0     |0.0       |
|0     |0.0       |
|1     |0.0       |
|0     |0.0       |
|1     |0.0       |
+------+----------+
only showing top 10 rows


In [56]:
results.printSchema()

root
 |-- features: vector (nullable = true)
 |-- Status: integer (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)



In [57]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [58]:
#confusion matrix
true_postives = results[(results.Status == 1) & (results.prediction == 1)].count()
true_negatives = results[(results.Status == 0) & (results.prediction == 0)].count()
false_positives = results[(results.Status == 0) & (results.prediction == 1)].count()
false_negatives = results[(results.Status == 1) & (results.prediction == 0)].count()

In [59]:
print (true_postives)
print (true_negatives)
print (false_positives)
print (false_negatives)
print(true_postives+true_negatives+false_positives+false_negatives)
print (results.count())

2326
2342
166
159
4993
4993


In [60]:
recall = float(true_postives)/(true_postives + false_negatives)
print(recall)

0.9360160965794768


In [61]:
precision = float(true_postives) / (true_postives + false_positives)
print(precision)

0.9333868378812199


In [62]:
accuracy=float((true_postives+true_negatives) /(results.count()))
print(accuracy)

0.93490887242139
